In [1]:

import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings("ignore")

In [2]:
CSV_PATH    = r"C:\Users\Stemadmin\Desktop\Anuj Khadka\carbon_footprint\results\benchmark_20260424_003610.csv"
OUTPUT_DIR  = Path(r"C:\Users\Stemadmin\Desktop\Anuj Khadka\carbon_footprint\results\analysis")
 
LANGUAGES   = ["c", "rust", "java", "go", "javascript", "python"]
ALGORITHMS  = ["summation", "binary_search", "merge_sort", "bfs", "hash_table", "matrix_multiplication"]
SIZES       = ["small", "mid", "large"]
 
LANG_LABELS = {
    "c": "C", "rust": "Rust", "java": "Java",
    "go": "Go", "javascript": "JavaScript", "python": "Python"
}
ALGO_LABELS = {
    "summation": "Summation", "binary_search": "Binary Search",
    "merge_sort": "Merge Sort", "bfs": "BFS",
    "hash_table": "Hash Table", "matrix_multiplication": "Matrix Mul"
}
SIZE_LABELS = {"small": "Small", "mid": "Mid", "large": "Large"}
 
# Colors per language — consistent across all charts
COLORS = {
    "c": "#2196F3",          # blue
    "rust": "#FF5722",       # deep orange
    "java": "#FF9800",       # orange
    "go": "#00BCD4",         # cyan
    "javascript": "#FFEB3B", # yellow
    "python": "#4CAF50",     # green
}

In [3]:
ALPHA_CORRECTED = 0.05 / 15   # Bonferroni correction
SEP = "=" * 65

In [4]:
def save(fig, name):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUTPUT_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {path.name}")

In [5]:
 
def cohens_d(a, b):
    pooled = np.sqrt((a.std()**2 + b.std()**2) / 2)
    return abs(a.mean() - b.mean()) / pooled if pooled > 0 else 0.0
 

In [6]:
print(SEP)
print("LOADING DATA")
print(SEP)
df = pd.read_csv(CSV_PATH)
print(f"  Rows           : {len(df)}")
print(f"  Columns        : {list(df.columns)}")
print(f"  Languages      : {sorted(df['language'].unique())}")
print(f"  Algorithms     : {sorted(df['algorithm'].unique())}")
print(f"  Sizes          : {sorted(df['size'].unique())}")
print(f"  Runs per cell  : {df.groupby(['language','algorithm','size'])['run'].count().unique()[0]}")
print(f"  Total joules   : {df['joules'].sum():.2f} J")
print(f"  Total gCO2e    : {df['gco2e'].sum():.6f} g")
print(f"  Carbon intensities: {sorted(df['carbon_intensity'].unique())}")
print(f"  Zero joules    : {(df['joules']==0).sum()}")
print(f"  Negative joules: {(df['joules']<0).sum()}")
 
lines = []   # collect all output for the summary TXT file
lines.append("RESULTS ANALYSIS SUMMARY")
lines.append(f"CSV: {CSV_PATH}\n")

LOADING DATA
  Rows           : 12420
  Columns        : ['language', 'algorithm', 'size', 'run', 'joules', 'kwh', 'carbon_intensity', 'gco2e', 'checksum']
  Languages      : ['c', 'go', 'java', 'javascript', 'python', 'rust']
  Algorithms     : ['bfs', 'binary_search', 'hash_table', 'matrix_multiplication', 'merge_sort', 'summation']
  Sizes          : ['large', 'mid', 'small']
  Runs per cell  : 115
  Total joules   : 86009.71 J
  Total gCO2e    : 7.278529 g
  Carbon intensities: [np.float64(300.0), np.float64(306.0)]
  Zero joules    : 0
  Negative joules: 0


In [ ]:
# ── SECTION 1: OVERALL MEAN gCO2e PER LANGUAGE ───────────────────────────────
print(f"\n{SEP}")
print("SECTION 1: OVERALL MEAN gCO2e PER LANGUAGE")
print(SEP)
 
lang_stats = df.groupby("language")["gco2e"].agg(
    mean="mean", std="std", median="median"
).loc[LANGUAGES]
c_mean = lang_stats.loc["c", "mean"]
 
lines.append(SEP)
lines.append("1. OVERALL MEAN gCO2e PER LANGUAGE (all algos, all sizes)")
lines.append(SEP)
print(f"  {'Language':<12} {'Mean gCO2e (g)':<20} {'Mean J':<12} {'Ratio to C'}")
print("  " + "-" * 55)
for lang in LANGUAGES:
    m   = lang_stats.loc[lang, "mean"]
    s   = lang_stats.loc[lang, "std"]
    j   = df[df["language"]==lang]["joules"].mean()
    rat = m / c_mean
    row = f"  {LANG_LABELS[lang]:<12} {m:.10f}     {j:.4f} J    {rat:.1f}x"
    print(row)
    lines.append(row)